## Emitter-Density Simulation — Image Generation

Generates 512×512 Bayer-camera images at 50 density levels (0.01–0.50 emitters/µm²)
for three dyes (ATTO 488, ATTO 565, ATTO 647N) and three photon levels (1000, 4000, 10 000).

Each (dye, n_photons, density) condition is saved as:
- A `(n_frames, 512, 512)` uint16 TIFF stack — independent noise realisations at that density
- A companion CSV with per-frame ground-truth emitter positions (x_px, y_px) and drawn photon counts

Save root: `/scratch/sycamore-asap/ASAP_Members_Other_Imaging_Data/JSB/Simulation/20260630_S3MEmitterDensity/`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import sys
import json
import time
import tifffile
from pathlib import Path

sys.path.append('../../..')
from src import IOFunctions, SpectralFunctions, MaskFunctions, PSFFunctions
from src.simulation import MultiC_Sim_Funcs

IO  = IOFunctions.IO_Functions()
S_F = SpectralFunctions.Spectral_Funcs()
M_F = MaskFunctions.Mask_Functions()
PSF = PSFFunctions.PSF_Functions()
MSF = MultiC_Sim_Funcs.MultiC_Sim_Funcs()

In [ ]:
# ── Camera calibration ─────────────────────────────────────────────────────────
data_folder = '../../../Camera_Calibrations/Ximea_Camera/'
gain      = IO.read_tiff(os.path.join(data_folder, 'gain.tif'))
offset    = IO.read_tiff(os.path.join(data_folder, 'offset.tif'))
variance  = IO.read_tiff(os.path.join(data_folder, 'variance.tif'))
readnoise = float(np.median(IO.read_tiff(os.path.join(data_folder, 'readnoise.tif'))))
rqe       = IO.read_tiff(os.path.join(data_folder, 'rqe.tif'))

gain_val     = float(np.median(gain))
offset_val   = float(np.median(offset))
variance_val = float(np.median(variance))
print(f'gain={gain_val:.4f}  offset={offset_val:.2f}  variance={variance_val:.4f}  readnoise={readnoise:.2f}')

In [ ]:
# ── Spectral setup ─────────────────────────────────────────────────────────────
R_base, G_base, B_base, wavelength = S_F.getpixelefficiency()
pixel_QYs = np.vstack([B_base, G_base, R_base])  # B=0, G=1, R=2
pixel_order = ['B', 'G', 'R']
filters = []

dyes   = ['ATTO 488', 'ATTO 565', 'ATTO 647N']
NA     = 1.49
pixel_size = 69.0   # nm/pixel

In [ ]:
# ── Simulation parameters ──────────────────────────────────────────────────────
image_size     = 512          # pixels (square)
n_frames       = 1000         # frames (realisations) per TIFF stack
bg_per_pixel   = 10.0          # background photoelectrons per pixel per frame

n_density_steps = 50
density_min     = 0.01        # emitters / µm²
density_max     = 0.50        # emitters / µm²
density_space   = np.linspace(density_min, density_max, n_density_steps)

n_photon_levels = [1000, 4000, 10_000]

# Image area in µm²
pixel_size_um   = pixel_size * 1e-3   # µm
image_area_um2  = (image_size * pixel_size_um) ** 2

print(f'Image: {image_size}×{image_size} px  ({image_size * pixel_size_um:.2f} µm × {image_size * pixel_size_um:.2f} µm)')
print(f'Image area: {image_area_um2:.1f} µm²')
print(f'Density range: {density_min}–{density_max} / µm²  ({n_density_steps} steps)')
print(f'N emitters at min/max density: {int(round(density_min * image_area_um2))} – {int(round(density_max * image_area_um2))}')
print(f'n_frames per stack: {n_frames}')
print(f'Total stacks: {len(density_space) * len(n_photon_levels) * len(dyes)} '
      f'({len(density_space)} densities × {len(n_photon_levels)} photon levels × {len(dyes)} dyes)')
total_gb = (
    n_density_steps * len(n_photon_levels) * len(dyes)
    * n_frames * image_size * image_size * 2 / 1e9
)
print(f'Estimated image data: {total_gb:.1f} GB')

In [ ]:
# ── 512×512 Bayer masks ────────────────────────────────────────────────────────
masks_512 = M_F.get_masks(size_x=image_size, size_y=image_size)
print('Mask channels:', list(masks_512.keys()))
for ch, m in masks_512.items():
    print(f'  {ch}: {m.sum()} True pixels  ({100 * m.mean():.1f}%)')

In [ ]:
# ── Save folder ────────────────────────────────────────────────────────────────
save_root = Path('/scratch/sycamore-asap/ASAP_Members_Other_Imaging_Data/JSB/Simulation/20260630_S3MEmitterDensity')
save_root.mkdir(parents=True, exist_ok=True)
print(f'Saving to: {save_root}')

In [ ]:
# ── camera_calibration dict for gen_camera_image_stack ────────────────────────
# Crop all calibration maps to image_size × image_size; gen_camera_image_stack
# derives the output image dimensions from gain.shape.
cam_gain     = gain[:image_size, :image_size].astype(np.float64)
cam_offset   = offset[:image_size, :image_size].astype(np.float64)
cam_variance = variance[:image_size, :image_size].astype(np.float64)
cam_rqe      = rqe[:image_size, :image_size].astype(np.float64)

camera_calibration_512 = {
    'gain':        cam_gain,
    'offset':      cam_offset,
    'variance':    cam_variance,
    'rqe':         cam_rqe,
    'pixel_order': pixel_order,
    'masks':       masks_512,
}
print(f'camera_calibration_512: {cam_gain.shape}  '
      f'gain_med={np.median(cam_gain):.4f}  offset_med={np.median(cam_offset):.2f}')

In [ ]:
# ── Pre-compute per-dye spectral parameters ────────────────────────────────────
# dpe_arr: raw (un-normalised) dye-pixel efficiency — used directly as Binomial
# probability p in gen_camera_image_stack (Binomial(photons_at_pixel, dpe[ch])).
dye_params = {}
for dye in dyes:
    aew, dpe_raw = S_F.get_pixel_fractions_dye_and_filters(
        [dye], filters, wavelength, pixel_QYs
    )
    dpe_arr  = np.array(dpe_raw).ravel()          # raw per-channel efficiency
    sigma_px = PSF.sigma_PSF(float(aew), NA) / pixel_size
    dye_params[dye] = {'dpe': dpe_arr, 'sigma_px': sigma_px, 'aew_nm': float(aew)}
    print(f'{dye:12s}  aew={float(aew):.0f} nm  sigma={sigma_px:.3f} px  dpe={np.round(dpe_arr, 3)}')

In [ ]:
# ── Save metadata ──────────────────────────────────────────────────────────────
meta = {
    'image_size':       image_size,
    'pixel_size_nm':    pixel_size,
    'NA':               NA,
    'n_frames':         n_frames,
    'bg_per_pixel':     bg_per_pixel,
    'density_space':    density_space.tolist(),
    'n_photon_levels':  n_photon_levels,
    'dyes':             dyes,
    'pixel_order':      pixel_order,
    'image_area_um2':   image_area_um2,
    'gain_val':         gain_val,
    'offset_val':       offset_val,
    'variance_val':     variance_val,
    'readnoise':        readnoise,
    'dye_params':       {
        d: {
            'dpe':      dye_params[d]['dpe'].tolist(),
            'sigma_px': dye_params[d]['sigma_px'],
            'aew_nm':   dye_params[d]['aew_nm'],
        }
        for d in dyes
    },
}
(save_root / 'metadata.json').write_text(json.dumps(meta, indent=2))
print(f'Saved metadata → {save_root / "metadata.json"}')

In [ ]:
# ── Main simulation loop ───────────────────────────────────────────────────────
#
# For each (dye, n_photons, density):
#   • generate n_frames independent 512×512 Bayer images via gen_camera_image_stack
#   • save as a (n_frames, 512, 512) uint16 TIFF stack
#   • save companion ground-truth CSV: frame_idx, emitter_id, x_px, y_px, n_photons
#
# Frames are generated in batches of BATCH to keep peak memory under ~1 GB.
# Each batch call: (BATCH, 512, 512) int64 × 2 intermediate arrays ≈ 400 MB.

BATCH = 100

n_total_cond = len(dyes) * len(n_photon_levels) * len(density_space)
i_cond       = 0
t0_total     = time.time()

for dye in dyes:
    dye_str = dye.replace(' ', '_')
    dp      = dye_params[dye]
    dpe     = dp['dpe']
    aew_nm  = dp['aew_nm']

    for n_photon in n_photon_levels:
        ph_dir = save_root / dye_str / f'{n_photon}ph'
        ph_dir.mkdir(parents=True, exist_ok=True)

        for d_idx, density in enumerate(density_space):
            i_cond     += 1
            N_emitters  = max(1, int(round(density * image_area_um2)))

            stem     = f'd{d_idx:02d}_density{density:.4f}'
            tif_path = ph_dir / f'{stem}_images.tif'
            gt_path  = ph_dir / f'{stem}_gt.csv'

            if tif_path.exists() and gt_path.exists():
                print(f'[{i_cond:4d}/{n_total_cond}] {dye:12s}  {n_photon:6d}ph  '
                      f'd{d_idx:02d}  N={N_emitters:4d}  (skipped)')
                continue

            t0_cond = time.time()
            gt_rows = []

            with tifffile.TiffWriter(tif_path, bigtiff=True) as tif:
                for b_start in range(0, n_frames, BATCH):
                    b_end  = min(n_frames, b_start + BATCH)
                    b_size = b_end - b_start

                    # Random emitter positions in nm for this batch
                    x_nm = np.random.uniform(0.0, image_size * pixel_size, (b_size, N_emitters))
                    y_nm = np.random.uniform(0.0, image_size * pixel_size, (b_size, N_emitters))
                    # x0y0[dye]: (b_size, 2, N_emitters) — positions in nm
                    x0y0_batch = {dye: np.stack([x_nm, y_nm], axis=1)}

                    bayer_batch, _, _ = MSF.gen_camera_image_stack(
                        camera_calibration_512,
                        wavelength,
                        aew_nm,
                        dpe,
                        {dye: np.full(b_size, float(n_photon))},
                        x0y0_batch,
                        smoothing_function=None,
                        background_photons=bg_per_pixel,
                        background_colour=[1.0] * len(pixel_order),
                        NA=NA,
                        pixel_size=pixel_size,
                    )
                    # gen_camera_image_stack squeezes size-1 dims; reshape defensively
                    bayer_batch = np.asarray(bayer_batch).reshape(b_size, image_size, image_size)

                    for frame_img in bayer_batch:
                        tif.write(frame_img.astype(np.uint16), contiguous=True)

                    for fi in range(b_size):
                        for em_id in range(N_emitters):
                            gt_rows.append({
                                'frame_idx':  b_start + fi,
                                'emitter_id': em_id,
                                'x_px':       float(x_nm[fi, em_id] / pixel_size),
                                'y_px':       float(y_nm[fi, em_id] / pixel_size),
                                'n_photons':  n_photon,
                            })

            pd.DataFrame(gt_rows).to_csv(gt_path, index=False)

            elapsed   = time.time() - t0_cond
            t_elapsed = (time.time() - t0_total) / 60
            t_remain  = (n_total_cond - i_cond) * elapsed / 60
            print(f'[{i_cond:4d}/{n_total_cond}] {dye:12s}  {n_photon:6d}ph  '
                  f'd{d_idx:02d}  N={N_emitters:4d}  '
                  f'{elapsed:.1f}s  elapsed={t_elapsed:.1f}m  ETA={t_remain:.1f}m')

print(f'\nAll done in {(time.time() - t0_total) / 60:.1f} min.')

In [ ]:
# ── Quick visual check: sample images at low, mid, high density ───────────────
check_dye      = 'ATTO 565'
check_n_photon = 4000
check_frame    = 0
check_d_idxs   = [0, 24, 49]   # first, middle, last density

fig, axs = plt.subplots(1, 3, figsize=(9, 3.5))

for ax, d_idx in zip(axs, check_d_idxs):
    density = density_space[d_idx]
    tif_path = (
        save_root
        / check_dye.replace(' ', '_')
        / f'{check_n_photon}ph'
        / f'd{d_idx:02d}_density{density:.4f}_images.tif'
    )
    if not tif_path.exists():
        ax.set_title(f'Missing: d{d_idx:02d}', fontsize=8)
        ax.axis('off')
        continue

    with tifffile.TiffFile(tif_path) as tif:
        frame = tif.pages[check_frame].asarray()

    n_em = max(1, int(round(density * image_area_um2)))
    vmin, vmax = np.percentile(frame, [1, 99])
    ax.imshow(frame, cmap='gray', vmin=vmin, vmax=vmax, origin='upper')
    ax.set_title(
        f'{density:.3f} /µm²  (N≈{n_em})', fontsize=8
    )
    ax.axis('off')

fig.suptitle(f'{check_dye}  {check_n_photon} ph  frame {check_frame}', fontsize=9)
fig.tight_layout()
plt.show()